# 03 — 유용한 대안을 보존하고 결과 증명하기

[English](03_frontiers_and_correctness.ipynb) | **한국어**

작은 프런티어가 불완전한 답을 조용히 반환하지 않고 정직하게 실패하게 만들어 봅시다. 그다음 독립 오라클과 몇 가지 요청 범위 실패를 살펴보겠습니다.

프런티어 연습은 스칼라 라우팅 엔진과 별개입니다. 이는 완전한 McRAPTOR 구현이나 승객 추천 정책이 **아닙니다**. [07장](../docs/07_multicriteria_frontiers_and_extensions.ko.md)을 읽어 보세요.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent
if not (root / "src" / "raptor.py").is_file():
    raise RuntimeError("Start this notebook from the repository root or notebooks directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from src import compile_timetable, demo_timetable, raptor, RoutingError
from src.pareto import Objective, bounded_frontier, dominates
fast = Objective("fast", departure=0, arrival=100, boardings=1, walking=30, access_burden=0, minimum_slack=30)
less_walk = Objective("less-walk", departure=0, arrival=110, boardings=1, walking=10, access_burden=0, minimum_slack=30)
assert not dominates(fast, less_walk)
assert not dominates(less_walk, fast)
frontier = bounded_frontier((fast, less_walk), capacity=2)
print("Preserved:", [item.journey_id for item in frontier])

## 중간 정류장에서 실제 경로 잃기
도착 시각과 도보량이 충돌하는 네트워크에서 실제 스칼라 라우터를 실행해 보세요. M에서는 한 번 탑승한 뒤 도착 시각 20이 21을 대체합니다. 명시적으로 열거한 도보량이 적은 여정은 여전히 가능하지만, 스칼라 목적지 결과를 정렬해서는 복구할 수 없습니다. 두 여정의 경로 증거는 모두 원시 입력으로 검사합니다.
이는 반례이며, 완전한 McRAPTOR나 다기준 오라클이 아닙니다. [반례 네트워크 풀이](../docs/07_multicriteria_frontiers_and_extensions.ko.md#run-a-path-loss-counterexample)를 참고하세요.

In [ ]:
from src.fixtures import walking_tradeoff_timetable
from example_walking_tradeoff import low_walking_witness
tradeoff = walking_tradeoff_timetable()
scalar = raptor(compile_timetable(tradeoff), "O", "Z", 0, max_boardings=2)
fast_path, = scalar.journeys()
low_path = low_walking_witness()
assert scalar.rows[1]["M"].time == 20
for label, path in (("scalar", fast_path), ("lost but feasible", low_path)):
    path.validate_against(tradeoff)
    print(label, path.arrival, path.boardings, path.walking_seconds)
assert (fast_path.arrival, fast_path.boardings, fast_path.walking_seconds) == (30, 2, 8)
assert (low_path.arrival, low_path.boardings, low_path.walking_seconds) == (35, 2, 1)
assert low_path not in scalar.journeys()


## 속이지 않고 용량 소진하기

공개 결과 상한은 내부 프런티어 정책이 아닙니다. 용량이 하나일 때 이 연습은 어느 비지배 목적 벡터도 잃지 않도록 거부합니다.

In [ ]:
try:
    bounded_frontier((fast, less_walk), capacity=1)
except RoutingError as error:
    assert error.code == "RAPTOR_FRONTIER_CAPACITY_EXCEEDED"
    print(error.code)
else:
    raise AssertionError("The frontier silently lost a required candidate.")

## 지배된 후보는 안전하게 사라져야 한다

느리고 도보 시간이 긴 후보는 바뀐 두 기준에서 모두 `fast`보다 나쁘며, 이를 보완할 이점도 없습니다. 이를 제거하는 것은 임의 절단이 아니라 지배 관계에 따른 결정입니다.

In [ ]:
dominated = Objective("dominated", departure=0, arrival=120, boardings=1, walking=40, access_burden=0, minimum_slack=30)
assert dominates(fast, dominated)
assert bounded_frontier((fast, dominated), capacity=1) == (fast,)
print("Dominated candidate removed; the useful candidate remains.")

## 승리한 카드뿐 아니라 모든 정류장 비교하기

라우팅 오라클은 `(stop, exact_boardings)` 상태를 사용하고 허용되는 모든 탑승/하차 가능성을 열거합니다. 아래 비교는 모든 최대 탑승 예산에서 모든 정류장을 검사합니다.

In [ ]:
from src.oracle import oracle_arrivals
index = compile_timetable(demo_timetable())
result = raptor(index, "O", "Z", 8 * 3600, max_boardings=3, boarding_slack=60)
expected = oracle_arrivals(index.timetable, "O", 8 * 3600, max_boardings=3, boarding_slack=60)
actual = tuple({stop: label.time for stop, label in row.items()} for row in result.rows)
assert actual == expected
print("All stop/budget labels agree with the independent oracle.")

## 유효한 형태가 유효한 생성은 아니다

아래 가짜 번들 ID는 식별자 일치를 보여 주기 위해서만 존재합니다. 오래되었거나 일치하지 않는 스냅샷을 마지막 성공 결과로 대체하면 안 됩니다. 이 장난감 오버레이는 서명을 검증하지도 실시간 정보 제공자 역할을 하지도 않습니다.

In [ ]:
from src.realtime import Snapshot, apply_snapshot
snapshot = Snapshot("toy-rt", "a" * 64, index.timetable.service_date, observed_at=100, valid_until=200)
try:
    apply_snapshot(index.timetable, snapshot, now=200)
except RoutingError as error:
    assert error.code == "REALTIME_STALE"
    print(error.code)
else:
    raise AssertionError("An expired snapshot was accepted.")

## 예산 실패는 관찰 가능한 결과다

이 실습에서 수행한 작업만 보고합니다. 메트릭에 가상의 제공자 호출 0이나 운영 지연 시간 주장을 추가하지 않습니다.

In [ ]:
from src.metrics import Work
try:
    raptor(index, "O", "Z", 8 * 3600, work=Work(max_units=1))
except RoutingError as error:
    assert error.code == "RAPTOR_WORK_CAPACITY_EXCEEDED"
    print(error.code)
else:
    raise AssertionError("The request exceeded its budget without failing.")
print("Measured successful-query operations:", result.metrics)

## 다음 실제 확장

완전한 다기준 라우팅을 만들려면 유용한 도보/여유 시간 라벨이 중간 정류장에서 사라지는 반례를 구성하세요. 독립적인 무제한 다기준 오라클을 추가한 뒤, 이에 맞는 상태 모음과 버전이 지정된 용량 정책을 도입하세요.

이 노트북은 로컬 모델만 검증합니다. 완전한 다기준 라우팅에는 중간 라벨 모음과 독립적인 다기준 오라클이 필요합니다. 여러 날짜 입력, 서명된 데이터 검증, 배포된 서비스는 이 구현의 범위 밖입니다.